In [1]:
from astropy import units as u
from astropy.coordinates import SkyCoord
from astropy.time import Time
from astroquery.jplhorizons import Horizons
from astroquery.mpc import MPC
from io import StringIO
import numpy as np
from scipy.interpolate import interp1d
from sora import Body, EphemPlanete, Observer
from sora.prediction import prediction

d:\Users\andre\Escritorio\Universal\Code\Python\projects\occ\.venv\Lib\site-packages\sora\body\shape\core.py:2: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


SORA version: 0.3.2


In [36]:
case1 = 'Silesia'

if isinstance(case1, list):
  print('List!')
else:
  print('Individual')

Individual


In [41]:
a = 'Hola'

a = [a]
a

['Hola']

In [22]:
rock = 'Silesia'
epoch = {
  "start": str(Time.now()),
  "stop": str(Time.now()),
  "step": "1m",
  "number": 1
}


eph = MPC.get_ephemeris(rock, start=epoch['start'], step=epoch['step'] + "in", number=epoch['number'])


In [23]:
eph_jpl = Horizons(id=rock, epochs=epoch)

In [24]:
eph_jpl.ephemerides()

targetname,datetime_str,datetime_jd,H,G,solar_presence,lunar_presence,RA,DEC,RA_app,DEC_app,RA_rate,DEC_rate,AZ,EL,AZ_rate,EL_rate,sat_X,sat_Y,sat_PANG,siderealtime,airmass,magextinct,V,surfbright,illumination,illum_defect,sat_sep,sat_vis,ang_width,PDObsLon,PDObsLat,PDSunLon,PDSunLat,SubSol_ang,SubSol_dist,NPole_ang,NPole_dist,EclLon,EclLat,r,r_rate,delta,delta_rate,lighttime,vel_sun,vel_obs,elong,elongFlag,alpha,lunar_elong,lunar_illum,sat_alpha,sunTargetPA,velocityPA,OrbPlaneAng,constellation,TDB-UT,ObsEclLon,ObsEclLat,NPole_RA,NPole_DEC,GlxLon,GlxLat,solartime,earth_lighttime,RA_3sigma,DEC_3sigma,SMAA_3sigma,SMIA_3sigma,Theta_3sigma,Area_3sigma,RSS_3sigma,r_3sigma,r_rate_3sigma,SBand_3sigma,XBand_3sigma,DoppDelay_3sigma,true_anom,hour_angle,alpha_true,PABLon,PABLat
---,---,d,mag,---,---,---,deg,deg,deg,deg,arcsec / h,arcsec / h,deg,deg,arcsec / min,arcsec / min,arcsec,arcsec,deg,h,---,mag,mag,mag / arcsec2,%,arcsec,arcsec,---,arcsec,deg,deg,deg,deg,deg,arcsec,deg,arcsec,deg,deg,AU,km / s,AU,km / s,min,km / s,km / s,deg,---,deg,deg,%,deg,deg,deg,deg,---,s,deg,deg,deg,deg,deg,deg,h,min,arcsec,arcsec,arcsec,arcsec,deg,arcsec2,arcsec,km,km / s,Hz,Hz,s,deg,h,deg,deg,deg
str21,str24,float64,float64,float64,str1,str1,float64,float64,float64,float64,float64,float64,int64,int64,int64,int64,float64,float64,float64,int64,int64,int64,float64,float64,float64,float64,float64,str1,float64,int64,int64,int64,int64,float64,float64,int64,int64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,str2,float64,float64,float64,float64,float64,float64,float64,str3,float64,float64,float64,int64,int64,float64,float64,int64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,int64,float64,float64,float64
257 Silesia (A886 GB),2026-Jul-09 18:27:07.261,2461231.268834035,9.68,0.15,,,29.592,10.63251,29.94612,10.76215,38.28158,14.99625,--,--,--,--,-263446.52,-41483.03,276.051,--,999,--,15.346,7.693,97.00469,0.001,273389.5,*,0.033767,--,--,--,--,69.49,0.01,--,--,11.3189,-1.3959,2.893249176664,-1.5628161,2.96686162584786,-23.7442444,24.67463574,18.1318377,34.1719946,75.9415,/L,19.9313,12.9,28.7741,83.7579,249.42,245.832,1.19129,Ari,69.183878,31.6254469,-1.3601057,--,--,148.42464,-48.898688,--,0.0,0.008,0.004,0.00894,0.00049,23.647,1.37e-05,0.009,3.7933,7e-07,0.01,0.02,2.5e-05,304.9093,--,19.9327,21.2848,-1.4003


In [95]:
# Hiperparámetros
rock = 'Silesia'
obs = 'W63'
epoch_jpl = {
  'start': '2019-06-27',
  'stop': '2019-06-29',
  'step': '1m'
}
epoch_mpc = {
  'start': '2019-06-27',
  'stop': '2019-06-29',
  'step': '1min'
}
mag_lim = 16
sora_step = 10 # Paso en segundos del muestreo de predicción de SORA

# Bajamos efemérides del asteroide con horizons
body_jpl = Horizons(id=rock, epochs=epoch_jpl, location=obs)
eph_jpl = body_jpl.ephemerides()
data_jpl = eph_jpl['datetime_jd', 'RA', 'DEC', 'delta'] # Datos que pide SORA en objeto efemérides
sigma_jpl = eph_jpl['RA_3sigma', 'DEC_3sigma'] # incertidumbre 3sigma para la ascención recta y declinación en ese instante
error_jpl = eph_jpl['SMAA_3sigma', 'SMIA_3sigma', 'Theta_3sigma'] # característica del elipse de error (matriz de covarianza diagonalizada)

# Bajamos efemérides del asteroide con MPCES
body_mpc = MPC.query_object('asteroid', name=rock)
eph_mpc = MPC.get_ephemeris(
  rock, 
  step=epoch_mpc['step'], 
  start=epoch_mpc['start'], 
  number=1441, 
  location=obs
)
eph_mpc['Date_jd'] = Time(eph_mpc['Date']).jd # Las fechas tienen que estar en formato juliano
data_mpc = eph_mpc['Date_jd', 'RA', 'Dec', 'Delta'] # Datos que pide SORA en objeto efemérides
error_mpc = eph_mpc['Uncertainty 3sig', 'Unc. P.A.'] # características del error principal (componente principal)

# Función para convertir tabla astropy de query a ephem planete de sora (ya permite predicciones con cualquier query)
def ephem_sora(name, data):
  df = data.to_pandas()
  buffer = StringIO()
  df.to_csv(buffer, sep=' ', header=False, index=False, float_format='%.12f')
  buffer.seek(0)
  return EphemPlanete(ephem=buffer, name=name)

# Instanciamos objeto de efemérides de JPL y MPC
eph_jpl_sora = ephem_sora(rock, data_jpl)
eph_mpc_sora = ephem_sora(rock, data_mpc)

# Variable controlada (objeto)
sora_body = Body(rock)

# Instanciamos objeto menor con SORA utilizando los datos de jpl
sora_body_jpl = Body(rock, ephem=eph_jpl_sora)
sora_body_mpc = Body(rock, ephem=eph_mpc_sora)

# Instanciamos observador
sora_obs = Observer(name=obs, code=obs)
jd = np.array(data_jpl['datetime_jd'])
ra = np.array(data_jpl['RA'])
dec = np.array(data_jpl['DEC'])
delta = np.array(data_jpl['delta'])


# Monkey patch
ra_interp = interp1d(jd, ra)
dec_interp = interp1d(jd, dec)
dist_interp = interp1d(jd, delta)

def get_position(time, observer='geocenter'):
    jd_query = np.atleast_1d(time.jd)

    return SkyCoord(
        ra_interp(jd_query)*u.deg,
        dec_interp(jd_query)*u.deg,
        distance=dist_interp(jd_query)*u.au
    )

eph_jpl_sora.get_position = get_position

Obtaining data for Silesia from SBDB
Obtaining data for Silesia from SBDB


d:\Users\andre\Escritorio\Universal\Code\Python\projects\occ\.venv\Lib\site-packages\sora\body\meta.py:372: UserWarning: spkid is different in Body (20000257) and EphemPlanete (None). Body's spkid will have higher priority
  warnings.warn('spkid is different in {0} ({1}) and {2} ({3}). {0}\'s spkid will have higher priority'.format(


Obtaining data for Silesia from SBDB


d:\Users\andre\Escritorio\Universal\Code\Python\projects\occ\.venv\Lib\site-packages\sora\body\meta.py:372: UserWarning: spkid is different in Body (20000257) and EphemPlanete (None). Body's spkid will have higher priority
  warnings.warn('spkid is different in {0} ({1}) and {2} ({3}). {0}\'s spkid will have higher priority'.format(


In [110]:
sora_pred = prediction(
  body=sora_body,
  reference_center=sora_obs,
  time_beg=Time(epoch_jpl['start']),
  time_end=Time(epoch_jpl['stop']),
  mag_lim=mag_lim,
  divs=3,
  radius=300,
  step=10,
  verbose=True
)

Ephemeris was split in 3 parts for better search of stars

Searching occultations in part 1/3
Generating Ephemeris between 2019-06-27 00:00:00.000 and 2019-06-27 15:59:50.000 ...
    46 GaiaDR3 stars downloaded
Identifying occultations ...

Searching occultations in part 2/3
Generating Ephemeris between 2019-06-27 16:00:00.000 and 2019-06-28 07:59:50.000 ...
    60 GaiaDR3 stars downloaded
Identifying occultations ...

Searching occultations in part 3/3
Generating Ephemeris between 2019-06-28 08:00:00.000 and 2019-06-28 23:59:50.000 ...
    53 GaiaDR3 stars downloaded
Identifying occultations ...

1 occultations found.


In [111]:
jpl_pred = prediction(
  body=sora_body_jpl,
  reference_center=sora_obs,
  time_beg=Time(epoch_jpl['start']),
  time_end=Time(epoch_jpl['stop']),
  mag_lim=mag_lim,
  divs=3,
  radius=300,
  step=10,
  verbose=True
)

Ephemeris was split in 3 parts for better search of stars

Searching occultations in part 1/3
Generating Ephemeris between 2019-06-27 00:00:00.000 and 2019-06-27 15:59:50.000 ...
    46 GaiaDR3 stars downloaded
Identifying occultations ...

Searching occultations in part 2/3
Generating Ephemeris between 2019-06-27 16:00:00.000 and 2019-06-28 07:59:50.000 ...
    60 GaiaDR3 stars downloaded
Identifying occultations ...

Searching occultations in part 3/3
Generating Ephemeris between 2019-06-28 08:00:00.000 and 2019-06-28 23:59:50.000 ...
    53 GaiaDR3 stars downloaded
Identifying occultations ...

No stellar occultation was found.


In [113]:
import inspect

print(inspect.getsource(type(eph_jpl_sora).fit_d2_ksi_eta))

    @deprecated_alias(log='verbose')  # remove this line in v1.0
    def fit_d2_ksi_eta(self, star, verbose=True):
        """Fits the projected position (orthographic projection) of the object in
        the tangent sky plane relative to a star.

        Parameters
        ----------
        star : `str`, `astropy.coordinates.SkyCoord`
            The coordinate of the star in the same reference frame as the ephemeris.

        verbose : `bool`, optional, default=True
            Enable log printing.
        """
        if type(star) == str:
            star = SkyCoord(star, unit=(u.hourangle, u.deg))
        if hasattr(self, 'star') and self.star.to_string('hmsdms', precision=5) == star.to_string('hmsdms',
                                                                                                  precision=5):
            return
        self.star = star
        target = self.ephem.transform_to(SkyOffsetFrame(origin=star))
        da = target.cartesian.y
        dd = target.cart

In [ ]:
from sora.prediction.core import occ_params

# usa el evento encontrado por sora_pred
print(sora_pred)

occ_params(
    star,
    sora_body_jpl.ephem,
    tiempo_evento,
    reference_center=sora_obs
)

def occ_params(star, ephem, time, n_recursions=5, max_tdiff=None, reference_center='geocenter'):
    """Calculates the parameters of the occultation, as instant, CA, PA.

    Parameters
    ----------
    star : `sora.Star`
        The coordinate of the star in the same reference frame as the ephemeris.
        It must be a Star object.

    ephem : `sora.Ephem*`
        Object ephemeris. It must be an Ephemeris object.

    time : `astropy.time.Time`
        Time close to occultation epoch to calculate occultation parameters.

    n_recursions : `int`, default=5
        The number of attempts to try obtain prediction parameters in case the
        event is outside the previous range of time.

    max_tdiff : `int`, default=None
        Maximum difference from given time it will attempt to identify the
        occultation, in minutes. If given, 'n_recursions' is ignored.

    reference_center : `str`, `sora.Observer`, `sora.Spacecraft`
            A SORA observer object or a string 'ge